# LogicRAG: query-time reasoning DAG demo

LogicRAG avoids constructing a graph over the entire corpus. Instead, it builds a **query-specific logic dependency graph** at inference time, topologically orders the subproblems, and resolves them in a dependency-aware sequence.

This notebook recreates that control flow with a tiny flat corpus, lexical retrieval, and deterministic extraction. No API key or external package is required.

Paper: https://arxiv.org/abs/2508.06105  
Official code: https://github.com/chensyCN/LogicRAG

In [ ]:
# A flat corpus: there is no pre-built corpus knowledge graph.
corpus = [
    {"doc_id": "D1", "text": "The scientist Lena Ortiz was born in Riverton."},
    {"doc_id": "D2", "text": "Lena Ortiz attended Northbridge University."},
    {"doc_id": "D3", "text": "Northbridge University is located in Alder City."},
    {"doc_id": "D4", "text": "Alder City is in the country of Norland."},
    {"doc_id": "D5", "text": "Riverton hosts an annual music festival."},
    {"doc_id": "D6", "text": "Southport College is located in Bay City."},
]

question = "Which country contains the university attended by the scientist born in Riverton?"
print(question)

## 1. Query logic dependency graph

In the full method, an LLM proposes these nodes and edges. Here they are explicit. Placeholders such as `{p1}` are filled with upstream answers before retrieval. The extraction patterns stand in for an answer model.

In [ ]:
nodes = {
    "p1": {
        "query_template": "Which scientist was born in Riverton?",
        "depends_on": [],
        "answer_pattern": r"scientist (.+?) was born in Riverton",
    },
    "p2": {
        "query_template": "Which university did {p1} attend?",
        "depends_on": ["p1"],
        "answer_pattern": r"{p1} attended (.+? University)",
    },
    "p3": {
        "query_template": "Which city is {p2} located in?",
        "depends_on": ["p2"],
        "answer_pattern": r"{p2} is located in (.+? City)",
    },
    "p4": {
        "query_template": "Which country contains {p3}?",
        "depends_on": ["p3"],
        "answer_pattern": r"{p3} is in the country of ([^.]+)",
    },
}
edges = [(dependency, node_id) for node_id, node in nodes.items() for dependency in node["depends_on"]]
print("Nodes:", list(nodes))
print("Edges:", edges)

In [ ]:
from collections import Counter, defaultdict, deque
import re

def topological_sort(nodes):
    indegree = {node_id: 0 for node_id in nodes}
    children = defaultdict(list)
    for node_id, node in nodes.items():
        for dependency in node["depends_on"]:
            if dependency not in nodes:
                raise ValueError(f"Unknown dependency {dependency!r} for {node_id!r}")
            children[dependency].append(node_id)
            indegree[node_id] += 1

    ready = deque(sorted(node_id for node_id, degree in indegree.items() if degree == 0))
    order = []
    while ready:
        node_id = ready.popleft()
        order.append(node_id)
        for child in sorted(children[node_id]):
            indegree[child] -= 1
            if indegree[child] == 0:
                ready.append(child)
    if len(order) != len(nodes):
        raise ValueError("The query dependency graph contains a cycle.")
    return order

execution_order = topological_sort(nodes)
print("Topological execution order:", execution_order)
assert execution_order == ["p1", "p2", "p3", "p4"]

## 2. Dependency-aware retrieval and resolution

Each subproblem is rendered only after its parents have been answered. Retrieval operates on the flat corpus. The supporting document—not every retrieved candidate—is added to rolling memory.

In [ ]:
STOPWORDS = {"a", "an", "the", "which", "what", "who", "did", "does", "is", "in", "of", "by"}

def tokens(text):
    return [token for token in re.findall(r"[a-z0-9]+", text.casefold()) if token not in STOPWORDS]

def retrieve(query, top_k=2):
    query_counts = Counter(tokens(query))
    ranked = []
    for document in corpus:
        document_counts = Counter(tokens(document["text"]))
        overlap = sum(min(count, document_counts[token]) for token, count in query_counts.items())
        ranked.append((overlap, document["doc_id"], document))
    return [item[2] for item in sorted(ranked, key=lambda item: (-item[0], item[1]))[:top_k]]

def render(template, answers, escape=False):
    values = {key: re.escape(value) if escape else value for key, value in answers.items()}
    return template.format(**values)

def resolve(node, answers):
    query = render(node["query_template"], answers)
    candidates = retrieve(query)
    pattern = re.compile(render(node["answer_pattern"], answers, escape=True), re.IGNORECASE)
    for document in candidates:
        match = pattern.search(document["text"])
        if match:
            return query, match.group(1), document, candidates
    raise LookupError(f"No answer found for {query!r} in retrieved candidates")

print("Retriever and resolver ready.")

In [ ]:
answers = {}
rolling_memory = []
trace = []

for node_id in execution_order:
    query, answer, evidence, candidates = resolve(nodes[node_id], answers)
    answers[node_id] = answer
    if evidence["doc_id"] not in {item["doc_id"] for item in rolling_memory}:
        rolling_memory.append(evidence)
    trace.append({
        "node_id": node_id,
        "rendered_query": query,
        "parent_answers": {parent: answers[parent] for parent in nodes[node_id]["depends_on"]},
        "retrieved_doc_ids": [item["doc_id"] for item in candidates],
        "supporting_doc_id": evidence["doc_id"],
        "answer": answer,
    })

for step in trace:
    print(step)
print("\nRolling evidence memory:")
for document in rolling_memory:
    print(f"  {document['doc_id']}: {document['text']}")

In [ ]:
# 3. Final composition from the terminal node.
final_answer = answers[execution_order[-1]]
print("Question:", question)
print("Answer:", final_answer)
assert final_answer == "Norland"
assert [step["supporting_doc_id"] for step in trace] == ["D1", "D2", "D3", "D4"]

## What the full method adds

The paper uses LLMs to create and adapt the dependency graph, answer subproblems, prune redundant graph nodes, summarize rolling context, check completeness, and generate the final answer. The essential distinction remains visible here: the graph is built over the **current query’s reasoning needs**, while the corpus stays as ordinary text.